In [ ]:
from ipywidgets import widgets
from ipywidgets import Layout

from IPython.display import display
import requests, json, time

from csi_client import CSI_Client


: 

In [ ]:
# Location of maestro

# maestro_ip = '10.0.0.64'
# maestro_ip = 'localhost'
maestro_ip = 'localhost'
# maestro_ip = '10.209.18.180'

maestro_port = '8090'

#Update for https!
maestro_url = 'http://' + maestro_ip + ':' + maestro_port

#Change to string with cert location to verify with https. ex: /etc/ssl/certs
cert_location = False

SERVICE_URL = maestro_url + '/service/csi_quintech_rf_matrix_0'

print(f"Service URL is {SERVICE_URL}")

maestro_keys = {'X-API-Key-CSI-Maestro-SystemOperator': 'SystemOperator-1',
                'X-API-Key-CSI-Maestro-Hub': 'Hub-1'}

get_headers = { 'accept':'application/json',
                'Content-Type': 'application/json', 
                **maestro_keys }


Service URL is http://localhost:8090/service/csi_make_model_0


In [3]:
ws_callback_output = widgets.Output(
    value='Websocket Messages',
    placeholder='Async msgs from websocket callback',
    description='Websocket:',
    layout=Layout(width='99%', height='350px', border='2px solid orange'),
    disabled=True
)

# @ws_callback_output.capture(widget)   # Note - capture() does not work on callbacks
def remote_event_handler(payload):
    # This function handles all remote messages coming over the websocket
    mws_banner = '[ Received message on websocket ]'
    ws_callback_output.append_display_data(mws_banner)
    ws_callback_output.append_display_data(payload)


# http response
http_response_output = widgets.Output(
    value='Response from HTTP POST',
    placeholder='Type something',
    description='HTTP\nResponse:',
    layout=Layout(width='99%', height='200px', border='2px solid blue'),
    disabled=True
)

@http_response_output.capture()
def home_post_action(event):
    http_response_output.clear_output()

    _payload = json.dumps(json.loads(json_input_textarea.value))
    print("Issuing HTTP POST with the following payload:")
    print(_payload)

    response = requests.post(SERVICE_URL, json={"payload": _payload}, headers=maestro_keys)
    print(f"response: {response}")
    
def output_clear_both(event):
    # clears both HTTP POST and Websocket output
    http_response_output.clear_output()
    ws_callback_output.clear_output()
    

def clear_output():
    change_output_button = widgets.Button(description="CLEAR")
    the_output = widgets.Output()
    clear_output_widget = widgets.VBox([change_output_button, the_output])
    clear_output_widget.click_count = 0

    def button_clicked(_button):
        clear_output_widget.click_count += 1
        the_output.clear_output()
        with the_output:
            print(f"button clicked {clear_output_widget.click_count} times.")

    change_output_button.on_click(button_clicked)

    return clear_output_widget


In [ ]:
# Create a CSI_Client, provide a function to handle websocket messages
try:
    rmtconn = CSI_Client(remote_event_handler, maestro_ip, maestro_port, use_https=cert_location)

    status = rmtconn.status()
    print(f'Maestro connection status is {status}')
except:
    print('Could not connect to maestro')


CSI_Client initialization
Maestro connection status is Up and running


In [ ]:
# Retrieve all DoF data from database
response = requests.get(SERVICE_URL, headers=get_headers, verify=cert_location)
print('Maestro response:')

print(json.dumps(response.json(),indent=4))

response_json = response.json()


Maestro response:
{
    "field_version_common": "common-2022.10.07",
    "description": "Template for creating a SNMP Based Device Proxy",
    "service_id": "ca315dc7-e611-4bac-af6e-9875c5bda444",
    "service_name": "csi_make_model_0",
    "landing_page": "http://192.168.200.63",
    "code_version": "V1.1",
    "debug_mode": [
        "REFLECTOR"
    ],
    "device": {
        "make": "Device Make",
        "model": "Device Model",
        "serial_number": "serial_number",
        "firmware_version": "firmware_version",
        "label": "Device Label",
        "comms": {
            "ip": "192.168.200.63",
            "read_community": "public",
            "write_community": "private",
            "snmp_version": "1"
        },
        "ip_address": "192.168.1.1"
    },
    "heartbeat": {
        "last_sent": "12:28:55 AM"
    },
    "log": {
        "escalation_levels": "INFO",
        "state": "HAS_ENTRIES",
        "clear_all_entries": null,
        "entries": {
            "16": 

In [8]:
# json input area
json_input_textarea = widgets.Textarea(
    value='',
    placeholder='{parameters: ...}',
    description='String:',
    layout=Layout(width='75%', height='100px'),
    disabled=False
)
json_input_textarea.description = "JSON"

# post button
home_post_button = widgets.Button(
    description='HTTP POST',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Update DoF values in JSON',
    layout=Layout(position='right'),
    icon='' # (FontAwesome names without the `fa-` prefix)
)
home_post_button.on_click(home_post_action)

# clear button
output_clear_button = widgets.Button(
    description='CLEAR OUTPUT',
    disabled=False,
    button_style='warning', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Clear output',
    layout=Layout(position='right'),
    icon='' # (FontAwesome names without the `fa-` prefix)
)
output_clear_button.on_click(output_clear_both)

top_right_box = widgets.VBox([home_post_button, output_clear_button])
top_box = widgets.HBox([json_input_textarea, top_right_box])

http_response_output_label = widgets.Label(value='HTTP Post Results')
ws_callback_output_label = widgets.Label(value='Websocket Callback Msgs')

bottom_box = widgets.VBox([http_response_output_label, http_response_output, 
                           ws_callback_output_label, ws_callback_output])
# bottom_box = widgets.VBox([http_response_output, ws_callback_output])

app_box = widgets.VBox([top_box, bottom_box])

app_box

# Some sample DoF state changes in json 
# Note: Use Ctrl-C and Ctrl-V to cut and paste in Jupyter

# {"proxy_configuration": {"comms": {"ip": "192.168.10.234", 
#  "read_community": "public", "write_community": "private", "snmp_version": 1}}}

# {"parameters": {"input_ports": {"2": {"description": "INPUT PORT 2"}}}}
# {"parameters": {"input_ports": {"3": {"gain": "33"}}}}

In [ ]:
# Walk inOut Table (using snmpwalk)
# Have to build net-snmp tools on C2 machine
# Enterprise mibs must be copied to /usr/local/share/snmp/mibs

# !MIBS=+EVERTZ-MINI-EXPERIMENTAL-NODE snmpwalk -v 1 -c private 192.168.10.234 1
!MIBS=ALL snmpwalk -v 2c -c public 192.168.1.248 quintech

Bad operator (INTEGER): At line 73 in /usr/share/snmp/mibs/ietf/SNMPv2-PDU
Timeout: No Response from X.X.X.X
